# [Main Quest 03] GPT-1 — Generative Pre-Training

번역기와 챗봇에서 구현한 Transformer를 **decoder-only 언어 모델**로 바꾸어,
OpenAI의 논문 *Improving Language Understanding by Generative Pre-Training* (2018)의
GPT-1 학습 프레임워크를 TensorFlow/Keras로 구현합니다.

이 노트북은 다음 요소를 포함합니다.

1. **GPT-1 decoder-only Transformer**: causal self-attention, learned position embedding, GELU
2. **사전학습 목적함수 \(L_1\)**: 다음 토큰 예측
3. **지도학습 목적함수 \(L_2\)**: 분류 레이블 예측
4. **보조 언어모델 목적함수 \(L_3=L_2+\lambda L_1\)**
5. 논문의 `<start>`, `<delimiter>`, `<extract>` 입력 변환
6. 분류·함의·유사도·객관식 문제에 재사용할 수 있는 task head

> 기본 설정은 CPU에서도 구조를 확인할 수 있는 작은 데모 모델입니다.
> 논문 크기(12 layers, 768 hidden, 12 heads)는 마지막 설정표를 참고하세요.

## 1. 번역기/챗봇 Transformer와 GPT-1의 차이

| 번역기·챗봇 | GPT-1 |
|---|---|
| Encoder + Decoder | **Decoder-only** Transformer |
| Encoder-Decoder cross-attention 사용 | cross-attention 없음 |
| 타겟 문장에서만 look-ahead mask | 모든 입력에 **causal mask** |
| sinusoidal positional encoding | **학습 가능한 position embedding** |
| 번역/응답 생성 목적 | 대규모 말뭉치 다음 토큰 예측 후 downstream fine-tuning |

논문의 최대화 목적함수는 다음과 같습니다.

\[
L_1(U)=\sum_i \log P(u_i\mid u_{i-k},\ldots,u_{i-1};\Theta)
\]

\[
P(y\mid x^1,\ldots,x^m)=\mathrm{softmax}(h_l^mW_y),\qquad
L_2(C)=\sum_{(x,y)}\log P(y\mid x^1,\ldots,x^m)
\]

\[
L_3(C)=L_2(C)+\lambda L_1(C)
\]

코드에서는 손실을 **최소화**하므로 부호가 바뀝니다.

\[
\mathcal{J}_{fine}
=\mathcal{J}_{classification}+\lambda\mathcal{J}_{LM}
\]

In [ ]:
# 필요한 라이브러리
# !pip install tensorflow-cpu==2.15.0 sentencepiece==0.2.0

import math
import os
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import sentencepiece as spm
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

## 2. 데모 말뭉치와 설정

실제 GPT-1은 BooksCorpus 약 7,000권을 사용했습니다. 여기서는 전체 파이프라인을 재현하기 위해
작은 한국어 말뭉치를 기본값으로 제공합니다. `GPT1_PRETRAIN_TEXT` 환경변수에 한 줄당 한 문서인
텍스트 파일 경로를 지정하면 외부 말뭉치로 교체할 수 있습니다.

In [ ]:
DEMO_CORPUS = [
    "트랜스포머는 어텐션을 사용해 문맥 속 토큰의 관계를 학습한다.",
    "언어 모델은 앞의 토큰을 조건으로 다음 토큰의 확률을 예측한다.",
    "GPT는 생성적 사전학습과 지도학습 미세조정을 결합한 모델이다.",
    "인과 마스크는 현재 위치가 미래 토큰을 보지 못하게 한다.",
    "멀티 헤드 어텐션은 서로 다른 표현 공간에서 정보를 모은다.",
    "잔차 연결과 층 정규화는 깊은 신경망의 학습을 안정화한다.",
    "위치 임베딩은 토큰의 순서 정보를 모델에 제공한다.",
    "사전학습된 언어 모델은 적은 레이블 데이터에도 일반화할 수 있다.",
    "분류 문제에서는 마지막 추출 토큰의 은닉 상태를 사용한다.",
    "보조 언어 모델 목적함수는 미세조정 중 과적합을 줄일 수 있다.",
    "자연어 추론은 전제와 가설 사이의 관계를 분류한다.",
    "의미 유사도 문제는 두 문장의 의미가 얼마나 가까운지 판단한다.",
    "객관식 문제는 문맥과 각 후보 답안을 하나의 시퀀스로 만든다.",
    "서브워드 토큰화는 자주 등장하는 문자열을 하나의 토큰으로 묶는다.",
    "생성 모델은 주어진 프롬프트 다음에 올 토큰을 반복해서 선택한다.",
    "좋은 표현은 여러 자연어 처리 과제에서 재사용할 수 있다.",
    "딥러닝 모델은 데이터와 목적함수에 따라 내부 표현을 최적화한다.",
    "학습률 워밍업은 훈련 초기에 급격한 파라미터 변화를 막는다.",
    "코사인 감쇠는 학습이 진행될수록 학습률을 부드럽게 낮춘다.",
    "GPT의 언어 모델 출력층은 입력 토큰 임베딩과 가중치를 공유한다.",
    "좋은 영화는 배우의 연기와 탄탄한 이야기로 관객을 몰입시킨다.",
    "아름다운 음악과 영상은 작품의 감동을 더 크게 만든다.",
    "지루한 전개와 어색한 연기는 영화에 집중하기 어렵게 한다.",
    "설득력 없는 결말은 관객에게 큰 실망을 줄 수 있다.",
    "재미있는 작품은 시간이 지나도 다시 보고 싶어진다.",
    "산만한 내용과 매력 없는 인물은 이야기의 힘을 떨어뜨린다.",
    "훌륭한 연출은 익숙한 소재도 새롭게 보여 줄 수 있다.",
    "영화 평가는 긍정과 부정 같은 감성 분류 문제로 만들 수 있다.",
]

pretrain_path = os.getenv("GPT1_PRETRAIN_TEXT")
if pretrain_path and Path(pretrain_path).exists():
    PRETRAIN_TEXTS = [
        line.strip()
        for line in Path(pretrain_path).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    PRETRAIN_TEXTS = DEMO_CORPUS

@dataclass(frozen=True)
class GPT1Config:
    vocab_size: int
    max_length: int = 64
    n_layers: int = 2
    d_model: int = 128
    n_heads: int = 4
    d_ff: int = 512
    dropout: float = 0.1

ARTIFACT_DIR = Path("artifacts/gpt1")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("사전학습 문서 수:", len(PRETRAIN_TEXTS))

## 3. SentencePiece BPE 토크나이저

GPT-1은 BPE를 사용했습니다. 여기서는 SentencePiece BPE로 같은 역할을 구현합니다.
downstream 입력 변환에 필요한 `<start>`, `<delimiter>`, `<extract>`도 사전에 추가합니다.

In [ ]:
corpus_path = ARTIFACT_DIR / "pretrain_corpus.txt"
corpus_path.write_text("\n".join(PRETRAIN_TEXTS), encoding="utf-8")
model_prefix = str(ARTIFACT_DIR / "gpt1_bpe")

if not Path(model_prefix + ".model").exists():
    spm.SentencePieceTrainer.train(
        input=str(corpus_path),
        model_prefix=model_prefix,
        vocab_size=256,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        user_defined_symbols=["<start>", "<delimiter>", "<extract>"],
        hard_vocab_limit=False,
    )

tokenizer = spm.SentencePieceProcessor(model_file=model_prefix + ".model")

PAD_ID = tokenizer.pad_id()
UNK_ID = tokenizer.unk_id()
BOS_ID = tokenizer.bos_id()
EOS_ID = tokenizer.eos_id()
START_ID = tokenizer.piece_to_id("<start>")
DELIMITER_ID = tokenizer.piece_to_id("<delimiter>")
EXTRACT_ID = tokenizer.piece_to_id("<extract>")
VOCAB_SIZE = tokenizer.vocab_size()

print(
    {
        "vocab": VOCAB_SIZE,
        "pad": PAD_ID,
        "bos": BOS_ID,
        "eos": EOS_ID,
        "start": START_ID,
        "delimiter": DELIMITER_ID,
        "extract": EXTRACT_ID,
    }
)
print(tokenizer.encode("생성적 사전학습을 구현합니다.", out_type=str))

## 4. 언어 모델 데이터셋

입력 시퀀스 자체를 한 칸 이동하여 정답으로 사용합니다.

- 입력: `[BOS, u₁, u₂, ..., uₙ₋₁]`
- 정답: `[u₁, u₂, ..., uₙ₋₁, EOS]`

In [ ]:
def build_lm_sequences(texts, max_length):
    sequences = []
    for text in texts:
        ids = [BOS_ID] + tokenizer.encode(text, out_type=int) + [EOS_ID]
        for start in range(0, len(ids) - 1, max_length):
            chunk = ids[start : start + max_length]
            if len(chunk) >= 2:
                sequences.append(chunk)
    return tf.keras.preprocessing.sequence.pad_sequences(
        sequences,
        maxlen=max_length,
        padding="post",
        truncating="post",
        value=PAD_ID,
    )

MAX_LENGTH = 64
BATCH_SIZE = 8
lm_sequences = build_lm_sequences(PRETRAIN_TEXTS, MAX_LENGTH)
lm_dataset = (
    tf.data.Dataset.from_tensor_slices(lm_sequences)
    .shuffle(len(lm_sequences), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("LM sequences:", lm_sequences.shape)
print("첫 시퀀스:", lm_sequences[0][:20])

## 5. GPT-1 decoder-only Transformer

기존 번역기/챗봇의 `MultiHeadAttention`과 `FeedForward`를 재사용하되 다음을 변경합니다.

- self-attention만 사용
- causal mask 적용
- learned token/position embedding
- ReLU 대신 GELU
- 언어 모델 출력 가중치와 token embedding 가중치 공유

원 논문 및 공개 코드와 같이 각 sub-layer 뒤에서 LayerNorm을 수행하는 post-norm 블록을 사용합니다.

### 마스킹 코드 리뷰

| 마스크 | 모양 | 역할 |
|---|---|---|
| `causal_mask` | `(seq, seq)` | query 위치보다 오른쪽의 미래 key를 차단해 자기회귀 조건을 보장합니다. |
| `key_mask` | `(batch, 1, 1, seq)` | `<pad>` key가 모든 query와 head의 attention에 참여하지 못하게 합니다. |
| `allowed` | `(batch, 1, seq, seq)` | 두 마스크가 모두 허용한 위치만 남기는 논리 AND 결과입니다. |

`tf.where`가 금지 위치의 score를 `-1e9`로 바꾸므로 softmax 뒤 가중치는 사실상 0이 됩니다.
Attention 마스크와 별도로 `language_model_loss`의 loss mask는 `<pad>` 정답이 gradient와 평균 손실에
포함되지 않게 합니다. 즉, **보지 못하게 하는 마스크**와 **채점하지 않는 마스크**를 각각 적용합니다.

In [ ]:
class CausalSelfAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model은 n_heads로 나누어져야 합니다.")
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = tf.keras.layers.Dense(3 * d_model)
        self.out = tf.keras.layers.Dense(d_model)
        self.attn_dropout = tf.keras.layers.Dropout(dropout)
        self.resid_dropout = tf.keras.layers.Dropout(dropout)

    def split_heads(self, x):
        batch_size = tf.shape(x)[0]
        seq_len = tf.shape(x)[1]
        x = tf.reshape(
            x, [batch_size, seq_len, self.n_heads, self.head_dim]
        )
        return tf.transpose(x, [0, 2, 1, 3])

    def call(self, x, attention_mask, training=False):
        q, k, v = tf.split(self.qkv(x), 3, axis=-1)
        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v)

        scale = tf.math.rsqrt(tf.cast(self.head_dim, tf.float32))
        scores = tf.matmul(q, k, transpose_b=True) * scale

        seq_len = tf.shape(x)[1]
        # causal_mask[q, k]는 k <= q일 때만 True: 미래 토큰 key를 차단합니다.
        causal_mask = tf.linalg.band_part(
            tf.ones([seq_len, seq_len], dtype=tf.bool), -1, 0
        )
        # (batch, seq) -> (batch, 1, 1, seq): batch/head/query 축으로 broadcast됩니다.
        key_mask = tf.cast(attention_mask[:, None, None, :], tf.bool)
        # 현재·과거 토큰이면서 실제 토큰인 위치만 attention 대상으로 허용합니다.
        allowed = causal_mask[None, None, :, :] & key_mask
        # 금지 위치는 softmax 전에 큰 음수로 바꿔 attention 확률을 0에 가깝게 만듭니다.
        scores = tf.where(
            allowed, scores, tf.cast(-1e9, scores.dtype)
        )

        weights = tf.nn.softmax(scores, axis=-1)
        weights = self.attn_dropout(weights, training=training)
        context = tf.matmul(weights, v)
        context = tf.transpose(context, [0, 2, 1, 3])
        context = tf.reshape(
            context, [tf.shape(x)[0], seq_len, self.d_model]
        )
        return self.resid_dropout(
            self.out(context), training=training
        )


class GPT1Block(tf.keras.layers.Layer):
    def __init__(self, config):
        super().__init__()
        self.attention = CausalSelfAttention(
            config.d_model, config.n_heads, config.dropout
        )
        self.ffn = tf.keras.Sequential(
            [
                tf.keras.layers.Dense(
                    config.d_ff, activation=tf.keras.activations.gelu
                ),
                tf.keras.layers.Dense(config.d_model),
                tf.keras.layers.Dropout(config.dropout),
            ]
        )
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)

    def call(self, x, attention_mask, training=False):
        attention_out = self.attention(
            x, attention_mask, training=training
        )
        x = self.norm1(x + attention_out)
        ffn_out = self.ffn(x, training=training)
        return self.norm2(x + ffn_out)


class GPT1Backbone(tf.keras.Model):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = tf.keras.layers.Embedding(
            config.vocab_size, config.d_model
        )
        self.position_embedding = tf.keras.layers.Embedding(
            config.max_length, config.d_model
        )
        self.embedding_dropout = tf.keras.layers.Dropout(config.dropout)
        self.blocks = [
            GPT1Block(config) for _ in range(config.n_layers)
        ]

    def call(self, input_ids, training=False):
        seq_len = tf.shape(input_ids)[1]
        tf.debugging.assert_less_equal(
            seq_len,
            self.config.max_length,
            message="입력 길이가 config.max_length보다 큽니다.",
        )
        positions = tf.range(seq_len)[None, :]
        hidden = self.token_embedding(input_ids)
        hidden += self.position_embedding(positions)
        hidden = self.embedding_dropout(hidden, training=training)
        # True는 실제 토큰, False는 <pad>이며 각 Transformer block에 전달됩니다.
        attention_mask = tf.not_equal(input_ids, PAD_ID)

        for block in self.blocks:
            hidden = block(
                hidden, attention_mask, training=training
            )

        # GPT-1: 입력 임베딩과 LM 출력 가중치를 공유(weight tying)
        lm_logits = tf.einsum(
            "btd,vd->btv",
            hidden,
            self.token_embedding.embeddings,
        )
        return hidden, lm_logits

## 6. 사전학습 목적함수 \(L_1\)

논문은 log-likelihood를 최대화합니다. 구현에서는 동일한 목적을
next-token cross entropy의 최소화로 표현하고, `<pad>` 위치는 손실에서 제외합니다.

In [ ]:
token_ce = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction="none"
)

def language_model_loss(input_ids, lm_logits):
    labels = input_ids[:, 1:]
    shifted_logits = lm_logits[:, :-1, :]
    # Attention mask와 별개로 <pad> 정답의 loss/gradient 기여를 제거합니다.
    mask = tf.cast(tf.not_equal(labels, PAD_ID), tf.float32)
    losses = token_ce(labels, shifted_logits) * mask
    return tf.math.divide_no_nan(
        tf.reduce_sum(losses), tf.reduce_sum(mask)
    )


class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps):
        super().__init__()
        self.peak_lr = peak_lr
        self.warmup_steps = max(1, warmup_steps)
        self.total_steps = max(self.warmup_steps + 1, total_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        total_steps = tf.cast(self.total_steps, tf.float32)
        warmup_lr = self.peak_lr * step / warmup_steps
        progress = tf.clip_by_value(
            (step - warmup_steps) / (total_steps - warmup_steps),
            0.0,
            1.0,
        )
        cosine_lr = 0.5 * self.peak_lr * (
            1.0 + tf.cos(math.pi * progress)
        )
        return tf.where(step < warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            "peak_lr": self.peak_lr,
            "warmup_steps": self.warmup_steps,
            "total_steps": self.total_steps,
        }


config = GPT1Config(vocab_size=VOCAB_SIZE, max_length=MAX_LENGTH)
gpt1 = GPT1Backbone(config)
_, sample_logits = gpt1(tf.constant(lm_sequences[:2]))
print("LM logits:", sample_logits.shape)
print("파라미터 수:", f"{gpt1.count_params():,}")

In [ ]:
PRETRAIN_EPOCHS = 5
pretrain_steps = max(1, int(tf.data.experimental.cardinality(lm_dataset)))
total_pretrain_steps = PRETRAIN_EPOCHS * pretrain_steps
pretrain_schedule = WarmupCosine(
    peak_lr=2.5e-4,
    warmup_steps=max(1, total_pretrain_steps // 10),
    total_steps=total_pretrain_steps,
)
pretrain_optimizer = tf.keras.optimizers.Adam(
    learning_rate=pretrain_schedule,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-8,
)

@tf.function
def pretrain_step(input_ids):
    with tf.GradientTape() as tape:
        _, logits = gpt1(input_ids, training=True)
        loss = language_model_loss(input_ids, logits)
    gradients = tape.gradient(loss, gpt1.trainable_variables)
    gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
    pretrain_optimizer.apply_gradients(
        zip(gradients, gpt1.trainable_variables)
    )
    return loss

for epoch in range(PRETRAIN_EPOCHS):
    losses = [pretrain_step(batch) for batch in lm_dataset]
    print(
        f"[Pre-train] epoch {epoch + 1:02d} | "
        f"L1 loss {tf.reduce_mean(losses):.4f}"
    )

checkpoint_path = ARTIFACT_DIR / "gpt1_demo_pretrained.weights.h5"
gpt1.save_weights(checkpoint_path)
print("사전학습 가중치 저장:", checkpoint_path)

## 7. 사전학습 모델로 텍스트 생성

매 시점에서 마지막 토큰의 분포만 사용해 다음 토큰을 샘플링합니다.

In [ ]:
SPECIAL_IDS = {
    PAD_ID,
    BOS_ID,
    EOS_ID,
    START_ID,
    DELIMITER_ID,
    EXTRACT_ID,
}

def generate(model, prompt, max_new_tokens=30, temperature=0.8, top_k=20):
    ids = [START_ID] + tokenizer.encode(prompt, out_type=int)
    for _ in range(max_new_tokens):
        model_input = ids[-config.max_length :]
        tensor = tf.constant([model_input], dtype=tf.int32)
        _, logits = model(tensor, training=False)
        next_logits = logits[:, -1, :] / temperature
        k = min(top_k, VOCAB_SIZE)
        values, indices = tf.math.top_k(next_logits, k=k)
        sampled = tf.random.categorical(values, num_samples=1)
        next_id = int(tf.gather(indices[0], sampled[0, 0]))
        if next_id in {EOS_ID, EXTRACT_ID}:
            break
        ids.append(next_id)

    decoded_ids = [token_id for token_id in ids if token_id not in SPECIAL_IDS]
    return tokenizer.decode(decoded_ids)

print(generate(gpt1, "트랜스포머는"))

## 8. GPT-1의 task-aware input transformation

논문은 모델 구조를 크게 바꾸는 대신 자연어 과제를 하나의 토큰 시퀀스로 변환합니다.

- 분류: `<start> text <extract>`
- 함의/문장쌍: `<start> premise <delimiter> hypothesis <extract>`
- 유사도: 두 문장 순서를 바꾼 시퀀스 2개를 각각 처리한 뒤 표현을 합산
- 객관식: 문맥과 각 후보를 각각 연결하여 후보별 점수 계산

In [ ]:
def truncate_content(token_ids, special_count, max_length=MAX_LENGTH):
    return token_ids[: max_length - special_count]

def format_classification(text):
    content = truncate_content(
        tokenizer.encode(text, out_type=int), special_count=2
    )
    return [START_ID] + content + [EXTRACT_ID]

def format_pair(text_a, text_b):
    budget = MAX_LENGTH - 3
    ids_a = tokenizer.encode(text_a, out_type=int)
    ids_b = tokenizer.encode(text_b, out_type=int)
    while len(ids_a) + len(ids_b) > budget:
        if len(ids_a) > len(ids_b):
            ids_a.pop()
        else:
            ids_b.pop()
    return [START_ID] + ids_a + [DELIMITER_ID] + ids_b + [EXTRACT_ID]

def format_similarity(text_a, text_b):
    return [format_pair(text_a, text_b), format_pair(text_b, text_a)]

def format_multiple_choice(context, choices):
    return [format_pair(context, choice) for choice in choices]

def pad_task_sequences(sequences):
    return tf.keras.preprocessing.sequence.pad_sequences(
        sequences,
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post",
        value=PAD_ID,
    )

print("분류:", format_classification("이 영화는 정말 재미있다."))
print("문장쌍:", format_pair("오늘 비가 온다.", "우산이 필요하다."))
print(
    "객관식 후보 수:",
    len(format_multiple_choice("하늘의 색은?", ["파랑", "초록", "검정"])),
)

## 9. 지도학습 목적함수 \(L_2\) + 보조목적함수 \(L_1\)

`<extract>` 위치의 마지막 Transformer block 은닉 상태 \(h_l^m\)에 선형 분류기를 연결합니다.
미세조정 중에는 분류 손실과 언어 모델 손실을 함께 최적화합니다.

\[
\mathcal{J}_{fine}
=\mathrm{CE}(y,\hat y)+\lambda\,
\mathrm{CE}(x_{2:m},\hat x_{2:m})
\]

In [ ]:
def gather_extract_hidden(hidden, input_ids):
    extract_mask = tf.equal(input_ids, EXTRACT_ID)
    tf.debugging.assert_equal(
        tf.reduce_sum(tf.cast(extract_mask, tf.int32), axis=1),
        tf.ones(tf.shape(input_ids)[0], dtype=tf.int32),
        message="각 입력에는 <extract>가 정확히 하나 있어야 합니다.",
    )
    positions = tf.argmax(
        tf.cast(extract_mask, tf.int32), axis=1, output_type=tf.int32
    )
    indices = tf.stack(
        [tf.range(tf.shape(input_ids)[0]), positions], axis=1
    )
    return tf.gather_nd(hidden, indices)


class GPT1ForSequenceClassification(tf.keras.Model):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.classifier = tf.keras.layers.Dense(num_classes)

    def call(self, input_ids, training=False):
        hidden, lm_logits = self.backbone(
            input_ids, training=training
        )
        pooled = gather_extract_hidden(hidden, input_ids)
        pooled = self.dropout(pooled, training=training)
        class_logits = self.classifier(pooled)
        return class_logits, lm_logits


SUPERVISED_SAMPLES = [
    ("이 영화는 배우들의 연기가 훌륭하고 매우 재미있다.", 1),
    ("스토리가 탄탄해서 끝까지 몰입해서 보았다.", 1),
    ("음악과 영상이 아름다워 다시 보고 싶다.", 1),
    ("따뜻하고 감동적인 작품이라 추천한다.", 1),
    ("전개가 지루하고 결말도 설득력이 없다.", 0),
    ("배우의 연기가 어색해서 집중하기 어려웠다.", 0),
    ("시간이 아까울 정도로 재미없는 영화였다.", 0),
    ("내용이 산만하고 등장인물도 매력이 없다.", 0),
]

task_inputs = pad_task_sequences(
    [format_classification(text) for text, _ in SUPERVISED_SAMPLES]
)
task_labels = np.asarray(
    [label for _, label in SUPERVISED_SAMPLES], dtype=np.int32
)
finetune_dataset = (
    tf.data.Dataset.from_tensor_slices((task_inputs, task_labels))
    .shuffle(len(task_labels), seed=SEED)
    .batch(4)
)

classifier = GPT1ForSequenceClassification(gpt1, num_classes=2)
_ = classifier(tf.constant(task_inputs[:2]))
print("분류기 포함 파라미터 수:", f"{classifier.count_params():,}")

In [ ]:
AUX_LM_WEIGHT = 0.5
FINETUNE_EPOCHS = 15
classification_ce = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)
finetune_optimizer = tf.keras.optimizers.Adam(
    learning_rate=6.25e-5,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-8,
)

@tf.function
def finetune_step(input_ids, labels):
    with tf.GradientTape() as tape:
        class_logits, lm_logits = classifier(
            input_ids, training=True
        )
        supervised_loss = classification_ce(labels, class_logits)
        auxiliary_lm_loss = language_model_loss(input_ids, lm_logits)
        total_loss = (
            supervised_loss + AUX_LM_WEIGHT * auxiliary_lm_loss
        )
    gradients = tape.gradient(
        total_loss, classifier.trainable_variables
    )
    gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
    finetune_optimizer.apply_gradients(
        zip(gradients, classifier.trainable_variables)
    )
    return total_loss, supervised_loss, auxiliary_lm_loss

for epoch in range(FINETUNE_EPOCHS):
    epoch_losses = [
        finetune_step(input_ids, labels)
        for input_ids, labels in finetune_dataset
    ]
    mean_losses = tf.reduce_mean(epoch_losses, axis=0)
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(
            f"[Fine-tune] epoch {epoch + 1:02d} | "
            f"L3 {mean_losses[0]:.4f} | "
            f"L2 {mean_losses[1]:.4f} | "
            f"aux L1 {mean_losses[2]:.4f}"
        )

In [ ]:
def predict_sentiment(text):
    input_ids = pad_task_sequences([format_classification(text)])
    logits, _ = classifier(tf.constant(input_ids), training=False)
    probabilities = tf.nn.softmax(logits, axis=-1)[0].numpy()
    return {
        "negative": float(probabilities[0]),
        "positive": float(probabilities[1]),
    }

print(predict_sentiment("연출이 훌륭하고 아주 재미있는 영화였다."))
print(predict_sentiment("지루하고 이해하기 어려워서 실망했다."))

## 10. 유사도·객관식 task head

아래 head는 논문 Figure 1의 입력 변환을 코드로 확장한 것입니다.

- `GPT1ForSimilarity`: 두 입력 순서의 `<extract>` 표현을 더한 뒤 분류
- `GPT1ForMultipleChoice`: 각 후보의 `<extract>` 표현을 scalar score로 변환

In [ ]:
class GPT1ForSimilarity(tf.keras.Model):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = tf.keras.layers.Dense(num_classes)

    def call(self, input_ids, training=False):
        # input_ids: (batch, 2, sequence)
        shape = tf.shape(input_ids)
        flat_ids = tf.reshape(input_ids, [-1, shape[2]])
        hidden, lm_logits = self.backbone(
            flat_ids, training=training
        )
        pooled = gather_extract_hidden(hidden, flat_ids)
        pooled = tf.reshape(pooled, [shape[0], 2, -1])
        combined = tf.reduce_sum(pooled, axis=1)
        return self.classifier(combined), lm_logits


class GPT1ForMultipleChoice(tf.keras.Model):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.scorer = tf.keras.layers.Dense(1)

    def call(self, input_ids, training=False):
        # input_ids: (batch, num_choices, sequence)
        shape = tf.shape(input_ids)
        flat_ids = tf.reshape(input_ids, [-1, shape[2]])
        hidden, lm_logits = self.backbone(
            flat_ids, training=training
        )
        pooled = gather_extract_hidden(hidden, flat_ids)
        choice_scores = self.scorer(pooled)
        choice_scores = tf.reshape(
            choice_scores, [shape[0], shape[1]]
        )
        return choice_scores, lm_logits


similarity_example = pad_task_sequences(
    format_similarity("고양이가 잠을 잔다.", "고양이가 자고 있다.")
)
similarity_example = tf.constant(similarity_example[None, ...])

choices = format_multiple_choice(
    "비가 올 때 필요한 물건은?", ["우산", "선글라스", "수영복"]
)
choice_example = tf.constant(
    pad_task_sequences(choices)[None, ...]
)

similarity_model = GPT1ForSimilarity(gpt1, num_classes=2)
multiple_choice_model = GPT1ForMultipleChoice(gpt1)
print("유사도 logits:", similarity_model(similarity_example)[0].shape)
print("객관식 logits:", multiple_choice_model(choice_example)[0].shape)

## 11. 논문 설정과 데모 설정

| 항목 | GPT-1 논문 | 이 노트북 기본값 |
|---|---:|---:|
| Transformer layers | 12 | 2 |
| hidden size | 768 | 128 |
| attention heads | 12 | 4 |
| FFN size | 3072 | 512 |
| context length | 512 | 64 |
| pre-training corpus | BooksCorpus | 작은 데모/사용자 텍스트 |
| pre-training batch | 64×512 tokens | 8 sequences |
| pre-training epochs | 100 | 5 |
| optimizer | Adam, warmup + cosine | 동일 계열 |

논문 규모 설정은 다음처럼 만들 수 있습니다.

```python
paper_config = GPT1Config(
    vocab_size=40000,
    max_length=512,
    n_layers=12,
    d_model=768,
    n_heads=12,
    d_ff=3072,
    dropout=0.1,
)
paper_gpt1 = GPT1Backbone(paper_config)
```

단, BooksCorpus 전체 학습은 GPU/TPU와 긴 학습 시간이 필요합니다. 이 노트북의 목적은
**GPT-1의 구조와 \(L_1\), \(L_2\), \(L_3\) 학습 프레임워크를 직접 확인하는 것**입니다.

## 핵심 정리

1. 번역기 Decoder에서 cross-attention을 제거하면 GPT 계열의 decoder-only 골격이 됩니다.
2. \(L_1\)으로 일반적인 언어 표현을 먼저 학습합니다.
3. downstream 데이터는 special token을 이용해 단일 시퀀스로 변환합니다.
4. \(L_2\)에 보조 \(L_1\)을 더한 \(L_3\)로 전체 모델과 task head를 함께 미세조정합니다.
5. 분류기 외에도 유사도·함의·객관식 head를 같은 backbone 위에 구성할 수 있습니다.

**참고**

- Radford et al., *Improving Language Understanding by Generative Pre-Training* (2018)
- OpenAI `finetune-transformer-lm` 공개 구현